### Задача 1

Анализ рынка вакансий

Предоставлен файл vacs.json, который содержит информацию о вакансиях, в частности, данные о предлагаемых зарплатах, даты размещения, описание

Предлагается ознакомиться с данными и найти:
- 10 самых новых вакансий
- 10 вакансий с максимальной годовой заработной платой
- среднюю зарплату
- все возможные типы занятости (полная занятость / совместительство)
- число вакансий, подходящих для старта (карьеры) и среднюю зарплату по ним
- число вакансий, требующих максимального опыта и среднюю зарплату по ним

Это задание лучше выполнять в Юпитер-ноутбуке построчно

In [16]:
import json
from pprint import pprint

with open("vacs.json", "r", encoding="utf-8") as f:
    data = json.load(f)


In [17]:
import datetime

pprint(sorted(data, key=lambda x: datetime.datetime.fromisoformat(x["data"]["add_date"]), reverse=True)[:10])


[{'data': {'add_date': '2023-01-20T16:55:22+03:00',
           'archive_date': None,
           'came_to_moderation_date': None,
           'currency': {'alias': 'RUB', 'id': 2996, 'title': '₽'},
           'description': '«Тинькофф» – это банк нового поколения, который '
                          'предлагает работу с удобным графиком - удаленную и '
                          'офисную.<br /><br />Общие требования к '
                          'кандидатам:<br /><br />- уверенный пользователь '
                          'ПК;<br /><br />- наличие устойчивого '
                          'интернет-соединения и головной гарнитуры с хорошим '
                          'микрофоном;<br /><br />- умение общаться с '
                          'людьми;<br /><br />- умение слушать и выполнять '
                          'задания руководителя;<br /><br />- опыт работы в '
                          'продажах приветствуется.<br /><br '
                          '/>«Тинькофф»предлагает:<br /><br />1. В

In [18]:
def get_salary(v):
    mx = v["data"]["salary_max"]
    mn = v["data"]["salary_min"]

    if mx is not None:
        return mx
    elif mn is not None:
        return mn
    else:
        return 0
        
pprint(sorted(data, key=get_salary, reverse=True)[:10])



[{'data': {'add_date': '2023-01-18T15:32:24+03:00',
           'archive_date': None,
           'came_to_moderation_date': None,
           'currency': {'alias': 'RUB', 'id': 2996, 'title': '₽'},
           'description': '<p><strong>В связи с расширением и потребностью '
                          'укрепить позицию на туристическом рынке перед '
                          'летним сезоном, туристическая компания TOURisME '
                          '(ТУРисМИ) ищет ведущего специалиста по реализации '
                          'топовых туристических продуктов. Ознакомьтесь, '
                          'пожалуйста, с требованиями к кандидатам, оцените те '
                          'выгоды, которые Вы получите, работая у нас, и '
                          'вышлите нам Ваше резюме!</strong></p> '
                          '<p><strong>Требования к кандидатам:</strong></p> '
                          '<p>Мы <strong>НЕ</strong> ищем рядового, пассивного '
                          'менеджера. 

In [19]:
from statistics import mean

def mean_salary(data):
    def get_salary(v):
        mx = v["data"]["salary_max"]
        mn = v["data"]["salary_min"]

        if mx is not None and mn is not None:
            return (mx + mn) / 2
        elif mx is not None:
            return mx
        elif mn is not None:
            return mn
        else:
            return 0

    return mean(get_salary(v) for v in data)

print(mean_salary(data))

68642.6975


In [20]:
pprint(set(v["data"]["working_type"]["title"] for v in data))

{'временная работа / freelance',
 'полная занятость',
 'работа вахтовым методом',
 'стажировка',
 'частичная занятость'}


In [21]:
print(set(v["data"]["experience_length"]["title"] for v in data))

{'3-6 лет', 'без опыта', 'более 6 лет', '1-3 года'}


In [22]:
print(mean_salary(v for v in data if v["data"]["experience_length"]["title"] == "без опыта"))
print(len([v for v in data if v["data"]["experience_length"]["title"] == "без опыта"]))

68882.62943262412
564


In [23]:
print(mean_salary(v for v in data if v["data"]["experience_length"]["title"] == "более 6 лет"))

116156.25


### Задача 2

Подсистема аутентификации

Требуется написать класс, который будет отвечать за аутентификацию пользователя в некоторой системе.

Класс должен содержать методы:
- для аутентификации пользователя по логину и паролю, возвращаемое значение типа bool (авторизован / нет)
- для добавления нового пользователя (пары логин-пароль)
- для смены пароля

Данные должны храниться перманентно, то есть не в памяти, а в каком-либо хранилище... для простоты можно использовать файл

Пароль должен быть защищён



In [24]:
import sqlite3
import hashlib


class Auth:
    def __init__(self, db_name="users.db"):
        self.conn = sqlite3.connect(db_name)
        self.cur = self.conn.cursor()

        self.cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                login TEXT PRIMARY KEY,
                password TEXT NOT NULL
            )
        """)
        self.conn.commit()


    def _hash(self, password):
        return hashlib.sha256(password.encode()).hexdigest()


    def add_user(self, login, password):
        self.cur.execute(
            "SELECT login FROM users WHERE login = ?",
            (login,)
        )
        if self.cur.fetchone():
            return False

        self.cur.execute(
            "INSERT INTO users (login, password) VALUES (?, ?)",
            (login, self._hash(password))
        )
        self.conn.commit()
        return True


    def authenticate(self, login, password):
        self.cur.execute(
            "SELECT password FROM users WHERE login = ?",
            (login,)
        )
        password_hash = self.cur.fetchone()[0]
        return password_hash == self._hash(password)


    def change_password(self, login, new_password):
        self.cur.execute(
            "SELECT login FROM users WHERE login = ?",
            (login,)
        )
        if not self.cur.fetchone():
            return False

        self.cur.execute(
            "UPDATE users SET password = ? WHERE login = ?",
            (self._hash(new_password), login)
        )
        self.conn.commit()
        return True


In [25]:
import time

auth = Auth()

auth.add_user("user1", "1234")

print(auth.authenticate("user1", "1234"))

time.sleep(3)

auth.change_password("user1", "4321")

print(auth.authenticate("user1", "4321"))

True
True
